In [1]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

from pathlib import Path

2026-08-23 11:57:28.528668: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-23 11:57:28.684511: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-23 11:57:28.684561: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-23 11:57:28.709416: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-23 11:57:28.762619: I tensorflow/core/platform/cpu_feature_guar

In [2]:
base_dir = Path("../dataset/pseudo_dataset")
IMG_SIZE = 256

def load_dataset(split):

    image_dir = base_dir / split / "images"
    mask_dir = base_dir / split / "masks"

    X = []
    Y = []

    for image_path in sorted(image_dir.glob("*")):

        mask_path = mask_dir / f"{image_path.stem}_mask.png"

        if not mask_path.exists():
            continue

        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

        mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE
        )

        mask = cv2.resize(
            mask,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_NEAREST
        )

        image = image.astype(np.float32) / 255.0

        mask = (mask > 127).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        X.append(image)
        Y.append(mask)

    return np.array(X), np.array(Y)

In [3]:
X_test, Y_test = load_dataset("test")

print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)

X_test: (44, 256, 256, 3)
Y_test: (44, 256, 256, 1)


In [ ]:
model = tf.keras.models.load_model(
    "../models/unet_dataset1_bce_dice_aug_best.keras",
    compile=False
)

print("Model loaded!")

2026-08-23 11:58:23.370452: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-23 11:58:23.480916: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-23 11:58:23.484486: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Model loaded!


In [6]:
predictions = model.predict(
    X_test,
    batch_size=1,
    verbose=1
)

pred_masks = (predictions >= 0.5).astype(np.float32)

2026-08-23 11:59:41.405541: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


44/44 [==============================] - 3s 21ms/step


In [8]:
def dice_score(y_true, y_pred, smooth=1e-6):
    y_true = y_true.astype(np.float32)
    y_pred = y_pred.astype(np.float32)

    intersection = np.sum(y_true * y_pred)

    return (
        2.0 * intersection + smooth
    ) / (
        np.sum(y_true) +
        np.sum(y_pred) +
        smooth
    )


def iou_score(y_true, y_pred, smooth=1e-6):
    y_true = y_true.astype(np.float32)
    y_pred = y_pred.astype(np.float32)

    intersection = np.sum(y_true * y_pred)

    union = (
        np.sum(y_true)
        + np.sum(y_pred)
        - intersection
    )

    return (
        intersection + smooth
    ) / (
        union + smooth
    )

In [9]:
dice_scores = []
iou_scores = []

for i in range(len(Y_test)):

    true_mask = Y_test[i].squeeze()
    pred_mask = pred_masks[i].squeeze()

    dice_scores.append(
        dice_score(true_mask, pred_mask)
    )

    iou_scores.append(
        iou_score(true_mask, pred_mask)
    )

dice_scores = np.array(dice_scores)
iou_scores = np.array(iou_scores)

print("Mean Dice:", dice_scores.mean())
print("Mean IoU :", iou_scores.mean())

print("Median Dice:", np.median(dice_scores))
print("Median IoU :", np.median(iou_scores))

Mean Dice: 0.8648188683209823
Mean IoU : 0.793189623807687
Median Dice: 0.9165681154628863
Median IoU : 0.8459930008289841


In [10]:
original_model = tf.keras.models.load_model(
    "../models/unet_bce_dice_aug_best.keras",
    compile=False
)

In [11]:
original_predictions = original_model.predict(
    X_test,
    batch_size=1,
    verbose=1
)

original_pred_masks = (
    original_predictions >= 0.5
).astype(np.float32)

44/44 [==============================] - 1s 22ms/step


In [12]:
original_dice_scores = []
original_iou_scores = []

for i in range(len(Y_test)):

    true_mask = Y_test[i].squeeze()
    pred_mask = original_pred_masks[i].squeeze()

    original_dice_scores.append(
        dice_score(true_mask, pred_mask)
    )

    original_iou_scores.append(
        iou_score(true_mask, pred_mask)
    )

original_dice_scores = np.array(original_dice_scores)
original_iou_scores = np.array(original_iou_scores)

print("Original Model")
print("----------------")
print("Mean Dice:", original_dice_scores.mean())
print("Mean IoU :", original_iou_scores.mean())
print("Median Dice:", np.median(original_dice_scores))
print("Median IoU :", np.median(original_iou_scores))

Original Model
----------------
Mean Dice: 0.8648188683209823
Mean IoU : 0.793189623807687
Median Dice: 0.9165681154628863
Median IoU : 0.8459930008289841
